<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/thermodynamics/modern_fluid_property_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Fluid Property Workflow

This notebook demonstrates a compact NeqSim workflow for creating a gas fluid, running a TP flash, reading thermodynamic and physical properties, and building a small property table.

In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

In [ ]:
import pandas as pd
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.thermo import TPflash, dataFrame

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos

## Create a Natural Gas Fluid

NeqSim's Java API uses Kelvin and bara by default. Set a mixing rule before running flashes.

In [ ]:
fluid = SystemSrkEos(273.15 + 25.0, 60.0)
fluid.addComponent("nitrogen", 0.01)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("methane", 0.86)
fluid.addComponent("ethane", 0.07)
fluid.addComponent("propane", 0.03)
fluid.addComponent("n-butane", 0.01)
fluid.setMixingRule("classic")
TPflash(fluid)
fluid.initProperties()
dataFrame(fluid).head()

## Read Key Properties

In [ ]:
properties = {
    "temperature_C": fluid.getTemperature("C"),
    "pressure_bara": fluid.getPressure("bara"),
    "density_kg_per_m3": fluid.getDensity("kg/m3"),
    "enthalpy_kJ_per_kg": fluid.getEnthalpy("kJ/kg"),
    "z_factor": fluid.getZ(),
}
pd.Series(properties, name="value")

## Generate a Property Table

The table below sweeps pressure at fixed temperature and records density and compressibility factor.

In [ ]:
rows = []
for pressure in [10, 30, 60, 90, 120]:
    case = fluid.clone()
    case.setPressure(float(pressure), "bara")
    TPflash(case)
    case.initProperties()
    rows.append({
        "pressure_bara": pressure,
        "density_kg_per_m3": case.getDensity("kg/m3"),
        "z_factor": case.getZ(),
    })

pd.DataFrame(rows)